In [17]:
!pip install transformers datasets torch --break-system-packages
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    pipeline,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)
import torch
from datasets import load_dataset
import argparse

Defaulting to user installation because normal site-packages is not writeable


In [18]:
def load_qa_model(model_name="deepset/roberta-base-squad2"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)
    return model, tokenizer

In [19]:
def load_summarization_model(model_name="facebook/bart-large-cnn"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    return model, tokenizer

In [20]:
def preprocess_squad(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [21]:
def train_qa(model, tokenizer, train_size=10000, eval_size=5000, epochs=1):
    squad = load_dataset("squad")
    tokenized_squad = squad.map(
        lambda examples: preprocess_squad(examples, tokenizer),
        batched=True,
        remove_columns=squad["train"].column_names
    )

    train_dataset = tokenized_squad["train"].select(range(train_size))
    eval_dataset = tokenized_squad["validation"].select(range(eval_size))

    training_args = TrainingArguments(
        output_dir="./results",
        evaluation_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=epochs,
        weight_decay=0.01,
        fp16=True,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=DefaultDataCollator(),
    )

    trainer.train()
    return model


In [22]:
def get_answer(qa_pipeline, question, context):
    qa_input = {
        'question': question,
        'context': context
    }
    return qa_pipeline(qa_input)

In [23]:
def summarize_text(summarizer, text, max_length=130, min_length=30):
    return summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)[0]['summary_text']

In [24]:
def main():
    qa_model, qa_tokenizer = load_qa_model()
    summarization_model, summarization_tokenizer = load_summarization_model()

    qa_pipeline = pipeline("question-answering", model=qa_model, tokenizer=qa_tokenizer, device=0 if torch.cuda.is_available() else -1)
    summarizer = pipeline("summarization", model=summarization_model, tokenizer=summarization_tokenizer, device=0 if torch.cuda.is_available() else -1)
    qa_pipeline.save_pretrained("./API/Model/qa_model")
    summarizer.save_pretrained("./API/Model/summ_model")


if __name__ == "__main__":
    main()

/home/anurag/.local/lib/python3.12/site-packages/transformers/modeling_utils.py:2618: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
